In [1]:
!pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\ianmu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os

os.makedirs("conf/model", exist_ok=True)
os.makedirs("conf/trainer", exist_ok=True)
os.makedirs("conf/logger", exist_ok=True)

config_yaml = """defaults:
  - model: cnn_scratch
  - trainer: default
  - logger: wandb

dataset:
  root: data
  class_name: ["bottle", "cable", "capsule", "grid", "pill", "screw", "tile", "toothbrush", "transistor", "zipper"]
  img_size: 128
  batch_size: 32
  num_classes: 2

train:
  lr: 1e-3
  epochs: 50
"""

with open("conf/config.yaml", "w") as f:
    f.write(config_yaml)

# Modelo A: CNN Scratch (ResNet-18 parcial entrenado desde cero)
model_a_yaml = """name: cnn_scratch
type: classifier
num_classes: ${dataset.num_classes}

architecture:
  base: resnet18_partial
  layers: [conv1, conv2_x, conv3_x]
  features_dim: 128

classifier:
  hidden_dim: 64
  dropout: 0.3

training:
  optimizer: adamw
  lr: ${train.lr}
  weight_decay: 1e-4
  scheduler: cosine
"""

with open("conf/model/cnn_scratch.yaml", "w") as f:
    f.write(model_a_yaml)

# Modelo B: Knowledge Distillation (ResNet-18 parcial con teacher-student)
model_b_yaml = """name: distilled
type: classifier
num_classes: ${dataset.num_classes}

architecture:
  base: resnet18_partial
  layers: [conv1, conv2_x, conv3_x]
  features_dim: 128

classifier:
  hidden_dim: 64
  dropout: 0.3

distillation:
  teacher_model: resnet18
  teacher_pretrained: true
  temperature: 4.0
  alpha: 0.7
  
training:
  optimizer: adamw
  lr: ${train.lr}
  weight_decay: 1e-4
  scheduler: cosine
"""

with open("conf/model/distilled.yaml", "w") as f:
    f.write(model_b_yaml)

# Modelo C: U-Net Autoencoder
model_c_yaml = """name: unet_ae
type: autoencoder

architecture:
  base: unet
  in_channels: 3
  base_channels: 64
  depth: 4

latent:
  dim: 256
  use_bottleneck: true

reconstruction:
  loss: mse
  perceptual_weight: 0.0
  use_ssim: false

training:
  optimizer: adam
  lr: 1e-4
  weight_decay: 1e-5
  scheduler: step
  scheduler_step: 20
  scheduler_gamma: 0.5
"""

with open("conf/model/unet_ae.yaml", "w") as f:
    f.write(model_c_yaml)

trainer_yaml = """max_epochs: ${train.epochs}
log_every_n_steps: 20
accelerator: gpu
devices: 1
precision: 16-mixed
"""

with open("conf/trainer/default.yaml", "w") as f:
    f.write(trainer_yaml)

logger_yaml = """project: proyecto2
entity: mauu-tec
log_model: true
"""

with open("conf/logger/wandb.yaml", "w") as f:
    f.write(logger_yaml)

print("Estructura de configuracion Hydra creada")
print("Modelos disponibles:")
print("  - cnn_scratch (Modelo A)")
print("  - distilled (Modelo B)")
print("  - unet_ae (Modelo C)")


Estructura de configuracion Hydra creada
Modelos disponibles:
  - cnn_scratch (Modelo A)
  - distilled (Modelo B)
  - unet_ae (Modelo C)


In [2]:
import wandb
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("WANDB_API_KEY")

if api_key:
    os.environ["WANDB_API_KEY"] = api_key
    wandb.login(key=api_key)
else:
    print("WANDB_API_KEY no está")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\ianmu\_netrc
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\ianmu\_netrc
wandb: Currently logged in as: mauu (mauu-tec) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Currently logged in as: mauu (mauu-tec) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
import torch

# Configurar precisión para Tensor Cores (API nueva)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"TF32 para matmul: {torch.backends.cuda.matmul.allow_tf32}")
print(f"TF32 para cuDNN: {torch.backends.cudnn.allow_tf32}")

GPU: NVIDIA GeForce RTX 3070 Ti
CUDA disponible: True
TF32 para matmul: True
TF32 para cuDNN: True


C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\backends\__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  self.setter(val)


In [7]:
import subprocess
import sys

def run_training(cmd):
    result = subprocess.run(cmd, encoding='utf-8', errors='replace')
    return result.returncode

run_training([sys.executable, "train_model_a.py", "train.lr=1e-3", "dataset.batch_size=32", "train.epochs=50"])
run_training([sys.executable, "train_model_a.py", "train.lr=5e-4", "dataset.batch_size=64", "train.epochs=40"])
run_training([sys.executable, "train_model_a.py", "train.lr=1e-4", "dataset.batch_size=32", "train.epochs=60"])

run_training([sys.executable, "train_model_b.py", "model=distilled", "train.lr=1e-3", "dataset.batch_size=32", "train.epochs=50", "model.distillation.temperature=4.0", "model.distillation.alpha=0.7"])
run_training([sys.executable, "train_model_b.py", "model=distilled", "train.lr=5e-4", "dataset.batch_size=64", "train.epochs=40", "model.distillation.temperature=3.0", "model.distillation.alpha=0.5"])
run_training([sys.executable, "train_model_b.py", "model=distilled", "train.lr=1e-4", "dataset.batch_size=32", "train.epochs=60", "model.distillation.temperature=5.0", "model.distillation.alpha=0.8"])

run_training([sys.executable, "train_model_c.py", "model=unet_ae", "train.lr=1e-3", "dataset.batch_size=32", "train.epochs=50", "model.latent.dim=256", "model.reconstruction.loss=l1"])
run_training([sys.executable, "train_model_c.py", "model=unet_ae", "train.lr=5e-4", "dataset.batch_size=64", "train.epochs=40", "model.latent.dim=128", "model.reconstruction.loss=l2"])
run_training([sys.executable, "train_model_c.py", "model=unet_ae", "train.lr=1e-4", "dataset.batch_size=32", "train.epochs=60", "model.latent.dim=512", "model.reconstruction.loss=l1"])

0